---
## Stage 10 v2: Dataset Building & Entity-Aware Split

**วัตถุประสงค์:** แบ่ง data เป็น train/val/test อย่างถูกต้อง (ไม่มี data leakage)

**Input:** `feature_matrix.csv`, `feature_cols.pkl` (จาก Stage 9 v2)  
**Output:** `scaler.pkl`, PyTorch DataLoaders

### ปรับปรุงจาก v1
- ใช้ `feature_matrix.csv` ที่มาจาก labeled pairs ที่ถูกต้อง (user_folder-based)
- เพิ่ม config cell ด้านบน
- entity split ใช้ `entity_id_a` ซึ่งตอนนี้เป็น user_folder จริงแล้ว

**กฎสำคัญ:**
- ห้าม random split pairs → entity เดียวกันจะอยู่ทั้ง train+test (leakage!)
- แบ่ง **entities** ก่อน → pairs ตามไป
- `scaler.fit(train)` เท่านั้น → `transform(val/test)`

| Sub-step | หน้าที่ |
|----------|--------|
| 10.1 | Entity-Aware Split (70/15/15) |
| 10.2 | Class Imbalance Handling |
| 10.3 | Feature Scaling (fit on train only!) |
| 10.4 | สร้าง PyTorch DataLoaders |

In [1]:
# ─── Config ───────────────────────────────────────────────────────────────
OUTPUT_DIR      = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed'
FEATURES_CSV    = f'{OUTPUT_DIR}/feature_matrix.csv'
FEAT_COLS_PKL   = f'{OUTPUT_DIR}/feature_cols.pkl'
SCALER_PKL      = f'{OUTPUT_DIR}/scaler.pkl'
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.15
TEST_RATIO      = 0.15
TRAIN_NEG_RATIO = 3
BATCH_SIZE      = 512
RANDOM_SEED     = 42
# ──────────────────────────────────────────────────────────────────────────

import os, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

feature_matrix = pd.read_csv(FEATURES_CSV)
with open(FEAT_COLS_PKL, 'rb') as f:
    feature_cols = pickle.load(f)
print(f'feature_matrix: {feature_matrix.shape} | features: {len(feature_cols)}')

feature_df = feature_matrix  # alias

feature_matrix: (204701, 21) | features: 17


### Step 10.1: Entity-Aware Train/Val/Test Split
แบ่ง **entities** (ไม่ใช่ pairs) → ป้องกัน data leakage  
`entity_id_a` = `user_folder` จาก Stage 8 v2

In [2]:
# --- 10.1 Entity-Aware Split ---

unique_entities = feature_df['entity_id_a'].dropna().unique()
rng = np.random.default_rng(RANDOM_SEED)
shuffled = unique_entities.copy()
rng.shuffle(shuffled)

n       = len(shuffled)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * (TRAIN_RATIO + VAL_RATIO))

train_entities = set(shuffled[:n_train])
val_entities   = set(shuffled[n_train:n_val])
test_entities  = set(shuffled[n_val:])

train_df = feature_df[feature_df['entity_id_a'].isin(train_entities)].copy()
val_df   = feature_df[feature_df['entity_id_a'].isin(val_entities)].copy()
test_df  = feature_df[feature_df['entity_id_a'].isin(test_entities)].copy()

print('📊 Step 10.1: Entity-Aware Split')
print('=' * 60)
print(f'  Total entities : {n:,}')
print(f'  Train entities : {len(train_entities):,}')
print(f'  Val entities   : {len(val_entities):,}')
print(f'  Test entities  : {len(test_entities):,}')
print(f'\n  Train pairs: {len(train_df):,} (pos={train_df["label"].sum():.0f}, neg={len(train_df)-train_df["label"].sum():.0f})')
print(f'  Val pairs  : {len(val_df):,} (pos={val_df["label"].sum():.0f}, neg={len(val_df)-val_df["label"].sum():.0f})')
print(f'  Test pairs : {len(test_df):,} (pos={test_df["label"].sum():.0f}, neg={len(test_df)-test_df["label"].sum():.0f})')

# Leakage check
overlap_tv = train_entities & val_entities
overlap_tt = train_entities & test_entities
overlap_vt = val_entities   & test_entities
print(f'\n  Leakage check:')
print(f'    Train∩Val  = {len(overlap_tv)} {"✅" if len(overlap_tv)==0 else "❌ LEAKAGE!"}')
print(f'    Train∩Test = {len(overlap_tt)} {"✅" if len(overlap_tt)==0 else "❌ LEAKAGE!"}')
print(f'    Val∩Test   = {len(overlap_vt)} {"✅" if len(overlap_vt)==0 else "❌ LEAKAGE!"}')
print(f'\n✅ Step 10.1 เสร็จ')

📊 Step 10.1: Entity-Aware Split
  Total entities : 15,176
  Train entities : 10,623
  Val entities   : 2,276
  Test entities  : 2,277

  Train pairs: 143,313 (pos=20476, neg=122837)
  Val pairs  : 30,660 (pos=4377, neg=26283)
  Test pairs : 30,713 (pos=4390, neg=26323)

  Leakage check:
    Train∩Val  = 0 ✅
    Train∩Test = 0 ✅
    Val∩Test   = 0 ✅

✅ Step 10.1 เสร็จ


/var/folders/ht/_lrx9n5s0539yfctp63z0yz00000gn/T/ipykernel_23414/2883095404.py:6: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(shuffled)


### Step 10.2: Class Imbalance Handling
Undersample negatives ใน train set / คำนวณ class weights

In [3]:
# --- 10.2 Class Imbalance ---

train_pos = train_df[train_df['label'] == 1]
train_neg = train_df[train_df['label'] == 0]
n_target_neg = len(train_pos) * TRAIN_NEG_RATIO

if len(train_neg) > n_target_neg:
    train_neg_sampled = train_neg.sample(n=n_target_neg, random_state=RANDOM_SEED)
    train_balanced = pd.concat([train_pos, train_neg_sampled]).sample(frac=1, random_state=RANDOM_SEED)
else:
    train_balanced = train_df.copy()

n_pos_b = (train_balanced['label'] == 1).sum()
n_neg_b = (train_balanced['label'] == 0).sum()
pos_weight = n_neg_b / max(n_pos_b, 1)

print('📊 Step 10.2: Class Imbalance Handling')
print('=' * 60)
print(f'  Before: pos={len(train_pos):,} neg={len(train_neg):,} ratio={len(train_neg)/max(len(train_pos),1):.1f}:1')
print(f'  After : pos={n_pos_b:,} neg={n_neg_b:,} ratio={n_neg_b/max(n_pos_b,1):.1f}:1')
print(f'  pos_weight = {pos_weight:.3f} (สำหรับ BCEWithLogitsLoss)')
print(f'\n✅ Step 10.2 เสร็จ')

📊 Step 10.2: Class Imbalance Handling
  Before: pos=20,476 neg=122,837 ratio=6.0:1
  After : pos=20,476 neg=61,428 ratio=3.0:1
  pos_weight = 3.000 (สำหรับ BCEWithLogitsLoss)

✅ Step 10.2 เสร็จ


### Step 10.3: Feature Scaling
`scaler.fit()` บน train เท่านั้น → `transform()` val+test

In [4]:
# --- 10.3 Feature Scaling ---
scaler = StandardScaler()

# FIT ON TRAIN ONLY!
X_train = scaler.fit_transform(train_balanced[feature_cols].values)
y_train = train_balanced['label'].values.astype(np.float32)

X_val  = scaler.transform(val_df[feature_cols].values)
y_val  = val_df['label'].values.astype(np.float32)

X_test = scaler.transform(test_df[feature_cols].values)
y_test = test_df['label'].values.astype(np.float32)

with open(SCALER_PKL, 'wb') as f:
    pickle.dump(scaler, f)

# Save test split metadata for stage12/13
test_df.to_csv(f'{OUTPUT_DIR}/test_dataset_formatted.csv', index=False)

print('📊 Step 10.3: Feature Scaling')
print('=' * 60)
print(f'  scaler.fit() บน train set เท่านั้น ({len(X_train):,} samples)')
print(f'  X_train: {X_train.shape} mean≈{X_train.mean():.4f} std≈{X_train.std():.4f}')
print(f'  X_val  : {X_val.shape}')
print(f'  X_test : {X_test.shape}')
print(f'  Saved: {SCALER_PKL}')
print(f'\n✅ Step 10.3 เสร็จ')

📊 Step 10.3: Feature Scaling
  scaler.fit() บน train set เท่านั้น (81,904 samples)
  X_train: (81904, 17) mean≈0.0000 std≈0.8745
  X_val  : (30660, 17)
  X_test : (30713, 17)
  Saved: /Users/tm/Documents/GitHub/Project-for-Work/data/processed/scaler.pkl

✅ Step 10.3 เสร็จ


### Step 10.4: สร้าง PyTorch DataLoaders

In [5]:
# --- 10.4 PyTorch DataLoaders ---
import torch
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    """Dataset สำหรับ feature-based pair classification"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):            return len(self.X)
    def __getitem__(self, idx):   return self.X[idx], self.y[idx]

train_dataset = PairDataset(X_train, y_train)
val_dataset   = PairDataset(X_val,   y_val)
test_dataset  = PairDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

sample_X, sample_y = next(iter(train_loader))

print('=' * 60)
print('📊 STAGE 10 v2 SUMMARY — Dataset Building')
print('=' * 60)
print(f'  Train loader : {len(train_loader)} batches ({len(train_dataset):,} samples)')
print(f'  Val loader   : {len(val_loader)} batches ({len(val_dataset):,} samples)')
print(f'  Test loader  : {len(test_loader)} batches ({len(test_dataset):,} samples)')
print(f'  Batch shape  : X={sample_X.shape}, y={sample_y.shape}')
print(f'  Input dim    : {sample_X.shape[1]} features')
print(f'  pos_weight   : {pos_weight:.3f}')
print(f'\n{"="*60}')
print(f'✅ Stage 10 v2 COMPLETE — DataLoaders พร้อมสำหรับ Training')
print(f'{"="*60}')

📊 STAGE 10 v2 SUMMARY — Dataset Building
  Train loader : 160 batches (81,904 samples)
  Val loader   : 60 batches (30,660 samples)
  Test loader  : 60 batches (30,713 samples)
  Batch shape  : X=torch.Size([512, 17]), y=torch.Size([512])
  Input dim    : 17 features
  pos_weight   : 3.000

✅ Stage 10 v2 COMPLETE — DataLoaders พร้อมสำหรับ Training
